In [2]:
from datasets import load_dataset

c:\Users\PC\.virtualenvs\deep-learning-1I3A7gMi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset('imdb')
small_train = dataset['train'].shuffle(seed=42).select([i for i in list(range(1000))])
small_test = dataset['test'].shuffle(seed=42).select([i for i in list(range(300))])

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_train.map(preprocess_function, batched=True)


In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 518.00it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir = './results',
    num_train_epochs = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    logging_dir = './logs',
    logging_steps = 10,
    report_to = 'none'
)

trainer = Trainer(
    model = model, 
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
c:\Users\PC\.virtualenvs\deep-learning-1I3A7gMi\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.701089
20,0.663018
30,0.682782
40,0.681764
50,0.506370
60,0.362767
70,0.416262
80,0.288102
90,0.328751
100,0.314497


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.91s/it]


TrainOutput(global_step=189, training_loss=0.3440191512385373, metrics={'train_runtime': 2363.1483, 'train_samples_per_second': 1.269, 'train_steps_per_second': 0.08, 'total_flos': 198701097984000.0, 'train_loss': 0.3440191512385373, 'epoch': 3.0})

In [7]:
trainer.evaluate()

c:\Users\PC\.virtualenvs\deep-learning-1I3A7gMi\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.08277160674333572,
 'eval_runtime': 247.2776,
 'eval_samples_per_second': 4.044,
 'eval_steps_per_second': 0.255,
 'epoch': 3.0}

In [16]:
texts =[ 
    "I really enjoyed this movie!",
    "This was a terrible film.",
    "The plot was predictable and boring.",
]

In [17]:
import torch
import torch.nn.functional as F

In [21]:
inputs = tokenizer(texts, padding = True, truncation = True, return_tensors = 'pt', max_length = 256)

with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim = 1)
    predictions = torch.argmax(probs, dim = 1)

label_map = {0: 'negative', 1: 'positive'}

for text, pred, prob in zip(texts, predictions, probs):
    print(text, label_map[pred.item()], prob[pred.item()].item())

I really enjoyed this movie! positive 0.9526634812355042
This was a terrible film. negative 0.9750536680221558
The plot was predictable and boring. negative 0.9837325215339661
